# Alzheimer's Disease Detection — 3D CNN on Pre-Fused Images
**Based on:** Kong et al., *Multi-modal data Alzheimer's disease detection based on 3D convolution*, Biomedical Signal Processing and Control, 2022

## Dataset structure
```
ADNI/
  ADNI_FUSED_FBB_AD/          ← MRI(MPRAGE) + PET(FBB)  fused  → AD
    019_S_10164/ fused_pet.nii.gz
    ...
  ADNI_FUSED_FBB_MCI/         ← MRI(MPRAGE) + PET(FBB)  fused  → MCI
    027_S_6788/  fused_pet.nii.gz
    ...
  ADNI_FUSED_OUTPUT_AD/       ← MRI(MPRAGE) + PET(AV45) fused  → AD
    007_S_10075/ fused_pet.nii.gz
    ...
  ADNI_FUSED_OUTPUT_CN/       ← MRI(MPRAGE) + PET(AV45) fused  → CN
    ...
  ADNI_FUSED_OUTPUT_MCI/      ← MRI(MPRAGE) + PET(AV45) fused  → MCI
    ...
```

**Expected totals:** AD≈32, MCI≈33, CN≈32 → ~97 subjects

## Pipeline
1. Mount Drive & scan ALL 5 folders
2. Preprocessing (crop → resize → normalize)
3. Sparse Autoencoder pre-training (KL sparsity, Eq. 1)
4. 3D CNN with SAE-initialized weights (Fig. 4)
5. 10-fold cross-validation → ACC, SEN, SPE (Eq. 2–4)
6. All 4 classification tasks from the paper

## Cell 1 — Install dependencies

In [ ]:
!pip install nibabel scikit-learn scipy -q
print("Dependencies installed.")

Dependencies installed.


## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted.")

Mounted at /content/drive
Drive mounted.


## Cell 3 — Imports & global settings

In [ ]:
import os
import numpy as np
import nibabel as nib
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from scipy.ndimage import zoom
from torch.utils.data import Dataset, DataLoader, Subset
from sklearn.model_selection import StratifiedKFold
from collections import Counter
import warnings
warnings.filterwarnings('ignore')

# ══════════════════════════════════════════════════════════════════════════════
#  ⚠️  UPDATE ROOT_DIR to match your Google Drive path
# ══════════════════════════════════════════════════════════════════════════════
ROOT_DIR = "/content/drive/MyDrive/ADNI"

# ── Device ────────────────────────────────────────────────────────────────────
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ── Paper Section 4.1: target volume size after preprocessing ─────────────────
TARGET_SHAPE = (121, 145, 121)   # fused PET/GM-PET target from the paper

# ── Paper Section 4.2: training hyperparameters ───────────────────────────────
EPOCHS      = 20     # paper: 200 epochs
BATCH_SIZE  = 3       # paper: batch size 3
LR          = 3e-6    # paper: 30 × 10⁻⁷ = 3 × 10⁻⁶
DROPOUT     = 0.5     # paper: dropout 0.5
FC_HIDDEN   = 80      # paper: 80 hidden units per FC layer (Fig. 4)
N_FOLDS     = 10      # paper: 10-fold cross-validation

# ── Sparse Autoencoder hyperparameters ────────────────────────────────────────
AE_EPOCHS   = 3      # pre-training epochs for SAE
AE_LR       = 1e-3    # SAE learning rate
KL_WEIGHT   = 0.1     # weight of sparsity penalty in loss
SPARSITY    = 0.05    # target sparsity ρ (Eq. 1)
N_FILTERS   = 32      # number of conv filters (paper uses 1 conv layer)
KERNEL_SIZE = 7       # 7×7×7 conv kernel (matches pool size in Fig. 4)

# ── Folder → class label mapping ──────────────────────────────────────────────
#
#   Both FBB (MRI+FBB-PET fused) and OUTPUT (MRI+AV45-PET fused) folders
#   contain the same type of data — a single fused NIfTI volume per subject.
#   They are combined into the same label classes.
#
FOLDER_TO_LABEL = {
    # MRI + FBB-PET fused images
    "ADNI_FUSED_FBB_AD":   0,   # AD  → 0
    "ADNI_FUSED_FBB_MCI":  1,   # MCI → 1
    # MRI + AV45-PET fused images
    "ADNI_FUSED_OUTPUT_AD":  0, # AD  → 0
    "ADNI_FUSED_OUTPUT_MCI": 1, # MCI → 1
    "ADNI_FUSED_OUTPUT_CN":  2, # CN  → 2
}

CLASS_NAMES = ["AD", "MCI", "CN"]

print(f"Device       : {DEVICE}")
print(f"Root dir     : {ROOT_DIR}")
print(f"Target shape : {TARGET_SHAPE}")
print(f"Epochs={EPOCHS} | LR={LR} | Batch={BATCH_SIZE} | "
      f"Dropout={DROPOUT} | FC={FC_HIDDEN}")
print(f"Filters={N_FILTERS} | Kernel={KERNEL_SIZE}×{KERNEL_SIZE}×{KERNEL_SIZE} "
      f"| Folds={N_FOLDS}")

## Cell 4 — Phase 1: Scan all 5 dataset folders

In [ ]:
def find_fused_nifti(subject_dir):
    """
    Find the fused NIfTI file inside a subject folder.
    Priority order: fused_pet.nii.gz → fused.nii.gz → any .nii.gz
    Returns the file path string, or None if not found.
    """
    d = Path(subject_dir)
    for candidate in ["fused_pet.nii.gz", "fused.nii.gz",
                       "fusion.nii.gz", "pet_fused.nii.gz"]:
        p = d / candidate
        if p.exists():
            return str(p)
    # Fall back: any .nii.gz in the folder
    nii_files = sorted(d.glob("*.nii.gz"))
    return str(nii_files[0]) if nii_files else None


def build_subject_list(root):
    """
    Walk all 5 ADNI folders (FBB + OUTPUT) and return a flat list:
      [{'path': str, 'label': int, 'subject_id': str,
        'modality': 'FBB'|'AV45', 'class': 'AD'|'MCI'|'CN'}, ...]

    Subjects from FBB and OUTPUT folders with the same label
    are pooled together — they are treated as the same class.
    """
    root = Path(root)
    subjects = []
    folder_summary = {}

    for folder_name, label in FOLDER_TO_LABEL.items():
        class_dir = root / folder_name

        if not class_dir.exists():
            print(f"  [WARNING] Folder not found: {class_dir}")
            folder_summary[folder_name] = 0
            continue

        # Determine modality from folder name
        modality = "FBB" if "FBB" in folder_name else "AV45"
        class_name = CLASS_NAMES[label]
        count = 0

        for subj_dir in sorted(class_dir.iterdir()):
            if not subj_dir.is_dir():
                continue
            nii_path = find_fused_nifti(subj_dir)
            if nii_path:
                subjects.append({
                    "path":       nii_path,
                    "label":      label,
                    "subject_id": subj_dir.name,
                    "modality":   modality,
                    "class":      class_name,
                })
                count += 1
            else:
                print(f"  [WARNING] No .nii.gz in {subj_dir}")

        folder_summary[folder_name] = count

    return subjects, folder_summary


# ── Scan dataset ──────────────────────────────────────────────────────────────
all_subjects, folder_summary = build_subject_list(ROOT_DIR)
all_labels = [s['label'] for s in all_subjects]
label_counts = Counter(all_labels)

print("\n── Folder-by-folder breakdown ──────────────────────────────")
for folder, count in folder_summary.items():
    modality = "FBB " if "FBB" in folder else "AV45"
    cls = folder.split("_")[-1]
    print(f"  {folder:<30}  [{modality}]  {count:>3} subjects")

print("\n── Combined class totals ───────────────────────────────────")
for label, name in enumerate(CLASS_NAMES):
    fbb_count  = sum(1 for s in all_subjects
                     if s['label'] == label and s['modality'] == 'FBB')
    av45_count = sum(1 for s in all_subjects
                     if s['label'] == label and s['modality'] == 'AV45')
    total      = fbb_count + av45_count
    print(f"  {name} (label={label}): {total:>3} total "
          f"({fbb_count} FBB + {av45_count} AV45)")

print(f"\n  TOTAL: {len(all_subjects)} subjects")

if len(all_subjects) == 0:
    print("\n❌ No subjects found! Check ROOT_DIR.")
else:
    print(f"\nSample entry (FBB): ",
          next((s for s in all_subjects if s['modality']=='FBB'), None))
    print(f"Sample entry (AV45):",
          next((s for s in all_subjects if s['modality']=='AV45'), None))

## Cell 5 — Phase 2: Preprocessing

In [ ]:
def load_nifti(path):
    """
    Load a NIfTI file as a float32 numpy array (always 3D).
    4-D volumes (PET time-series) are collapsed by mean over time axis.
    """
    data = nib.load(path).get_fdata().astype(np.float32)
    if data.ndim == 4:
        data = np.mean(data, axis=3)
    return data


def crop_background(vol, threshold=0.0):
    """
    Tight bounding box around non-zero voxels.
    Removes zero-padded background introduced by registration.
    Paper Section 4.1.
    """
    coords = np.argwhere(vol > threshold)
    if coords.size == 0:
        return vol
    x0, y0, z0 = coords.min(axis=0)
    x1, y1, z1 = coords.max(axis=0) + 1
    return vol[x0:x1, y0:y1, z0:z1]


def resize_volume(vol, target_shape):
    """
    Trilinear resample to target_shape using scipy zoom (order=1).
    Same approach as the paper's cropping/sampling step.
    """
    factors = [t / s for t, s in zip(target_shape, vol.shape)]
    return zoom(vol, factors, order=1).astype(np.float32)


def minmax_normalize(vol):
    """
    Scale voxel intensities to [0, 1].
    Applied to fused PET volumes (non-Gaussian distribution).
    """
    vmin, vmax = vol.min(), vol.max()
    return ((vol - vmin) / (vmax - vmin + 1e-8)).astype(np.float32)


def preprocess(path):
    """
    Full preprocessing for one fused volume:
      load → collapse 4D → crop background → resize → minmax normalize
    Returns float32 array of shape TARGET_SHAPE.
    """
    vol = load_nifti(path)
    vol = crop_background(vol)
    vol = resize_volume(vol, TARGET_SHAPE)
    vol = minmax_normalize(vol)
    return vol


# ── Verify on both modalities ─────────────────────────────────────────────────
for modality in ['FBB', 'AV45']:
    sample = next((s for s in all_subjects if s['modality'] == modality), None)
    if sample:
        vol = preprocess(sample['path'])
        print(f"[{modality}] {sample['subject_id']} ({sample['class']})")
        print(f"   shape={vol.shape}  range=[{vol.min():.4f}, {vol.max():.4f}]")
    else:
        print(f"[{modality}] No sample found.")

print("✅ Preprocessing OK")

## Cell 6 — PyTorch Dataset

In [ ]:
class FusedDataset(Dataset):
    """
    PyTorch Dataset for pre-fused ADNI brain volumes.
    Works identically for FBB and AV45 fused images — both are
    single-channel 3D volumes representing GM-masked PET activity.

    Returns:
      tensor : (1, D, H, W) float32  — channel-first for Conv3d
      label  : int  (0=AD, 1=MCI, 2=CN)
    """

    def __init__(self, subjects):
        self.subjects = subjects

    def __len__(self):
        return len(self.subjects)

    def __getitem__(self, idx):
        s   = self.subjects[idx]
        vol = preprocess(s['path'])                    # (D, H, W)
        t   = torch.from_numpy(vol).unsqueeze(0)       # (1, D, H, W)
        return t, s['label']


# ── Build the full dataset ────────────────────────────────────────────────────
full_dataset = FusedDataset(all_subjects)
x, y = full_dataset[0]
print(f"Dataset size : {len(full_dataset)} subjects")
print(f"Tensor shape : {x.shape}   dtype={x.dtype}")
print(f"Label        : {y} ({CLASS_NAMES[y]})")

# Verify one FBB and one AV45 subject load correctly
fbb_idx  = next((i for i,s in enumerate(all_subjects) if s['modality']=='FBB'), None)
av45_idx = next((i for i,s in enumerate(all_subjects) if s['modality']=='AV45'), None)

for name, idx in [('FBB', fbb_idx), ('AV45', av45_idx)]:
    if idx is not None:
        x2, y2 = full_dataset[idx]
        print(f"  [{name}] tensor shape={x2.shape}  label={y2} "
              f"({CLASS_NAMES[y2]})  range=[{x2.min():.3f},{x2.max():.3f}]")

print("✅ Dataset ready")

## Cell 7 — Phase 4: Sparse Autoencoder (Paper Section 3.3.1, Eq. 1)

In [ ]:
class SparseAutoencoder3D(nn.Module):
    """
    3D Sparse Autoencoder — Section 3.3.1 of the paper.

    Encoder: Conv3d(1 → N_FILTERS, KERNEL_SIZE) + ReLU
    Decoder: ConvTranspose3d(N_FILTERS → 1, KERNEL_SIZE) + Sigmoid

    After training, encoder weights are copied into the CNN's
    first convolutional layer as described in Section 3.3.2.
    """

    def __init__(self, n_filters=N_FILTERS, kernel_size=KERNEL_SIZE):
        super().__init__()
        pad = kernel_size // 2
        self.encoder = nn.Sequential(
            nn.Conv3d(1, n_filters, kernel_size=kernel_size, padding=pad),
            nn.ReLU()
        )
        self.decoder = nn.Sequential(
            nn.ConvTranspose3d(n_filters, 1, kernel_size=kernel_size, padding=pad),
            nn.Sigmoid()
        )

    def forward(self, x):
        hidden  = self.encoder(x)                   # (B, n_filters, D, H, W)
        recon   = self.decoder(hidden)              # (B, 1, D, H, W)
        # Mean activation per filter — shape (n_filters,)
        rho_hat = hidden.mean(dim=[0, 2, 3, 4])
        return recon, rho_hat

    def encoder_weights(self):
        conv = self.encoder[0]
        return conv.weight.data.clone(), conv.bias.data.clone()


def kl_divergence(rho, rho_hat):
    """
    KL divergence between Bernoulli(rho) and Bernoulli(rho_hat).
    Equation (1) from the paper — added to MSE loss as sparsity penalty.
    """
    rho     = torch.tensor(rho, dtype=torch.float32, device=rho_hat.device)
    rho_hat = rho_hat.clamp(1e-8, 1 - 1e-8)
    return (rho * torch.log(rho / rho_hat) +
            (1 - rho) * torch.log((1 - rho) / (1 - rho_hat))).mean()


def train_sae(train_loader, verbose=True):
    """
    Pre-train the sparse autoencoder on the current training fold.
    Loss = MSE(reconstruction, input) + KL_WEIGHT * KL(rho || rho_hat)
    Returns the trained SAE.
    """
    sae = SparseAutoencoder3D().to(DEVICE)
    opt = torch.optim.Adam(sae.parameters(), lr=AE_LR)
    mse = nn.MSELoss()

    for epoch in range(AE_EPOCHS):
        sae.train()
        total = 0.0
        for x, _ in train_loader:
            x = x.to(DEVICE)
            recon, rho_hat = sae(x)
            loss = mse(recon, x) + KL_WEIGHT * kl_divergence(SPARSITY, rho_hat)
            opt.zero_grad()
            loss.backward()
            opt.step()
            total += loss.item()
        if verbose and (epoch + 1) % 5 == 0:
            print(f"    [SAE] Epoch {epoch+1:2d}/{AE_EPOCHS}  "
                  f"loss={total/max(len(train_loader),1):.5f}")
    return sae


# ── Architecture check ────────────────────────────────────────────────────────
sae_chk = SparseAutoencoder3D()
dummy   = torch.zeros(1, 1, *TARGET_SHAPE)
r, rh   = sae_chk(dummy)
print(f"SAE encoder weights : {sae_chk.encoder[0].weight.shape}")
print(f"SAE input  : {dummy.shape}")
print(f"SAE recon  : {r.shape}")
print(f"rho_hat    : {rh.shape}  (one per filter)")
del sae_chk, dummy, r, rh
print("✅ SAE OK")

## Cell 8 — Phase 5: 3D CNN (Paper Section 3.3.2 / Fig. 4)

In [ ]:
class CNN3D(nn.Module):
    """
    3D CNN exactly matching Fig. 4 of the paper.

    Architecture:
      Input (1, 121, 145, 121)
        → Conv3d(1→32, 7×7×7, pad=3)   [weights from SAE encoder]
        → ReLU
        → MaxPool3d(7×7×7, stride=7)    [paper Fig. 4 label: '7x7x7 Max-pooling']
        → Flatten
        → Linear(flat → 80) → ReLU → Dropout(0.5)
        → Linear(80 → 80)   → ReLU → Dropout(0.5)
        → Linear(80 → n_classes)
        → LogSoftmax

    After MaxPool on (121,145,121) with stride 7:
      D=17, H=20, W=17  →  flat = 32×17×20×17 = 184,960
    """

    def __init__(self,
                 n_classes    = 3,
                 n_filters    = N_FILTERS,
                 kernel_size  = KERNEL_SIZE,
                 hidden_units = FC_HIDDEN,
                 dropout_p    = DROPOUT,
                 input_shape  = TARGET_SHAPE):
        super().__init__()

        pad = kernel_size // 2

        # ── 1 convolutional layer (SAE-initialised) ───────────────────────────
        self.conv = nn.Conv3d(1, n_filters,
                              kernel_size=kernel_size, padding=pad)

        # ── 7×7×7 max-pool as in Fig. 4 ──────────────────────────────────────
        self.pool = nn.MaxPool3d(kernel_size=7, stride=7)

        # ── Flattened feature size ────────────────────────────────────────────
        D, H, W = input_shape
        self.flat_size = n_filters * (D // 7) * (H // 7) * (W // 7)

        # ── 2 FC layers, 80 hidden units each (paper Fig. 4) ─────────────────
        self.fc1   = nn.Linear(self.flat_size, hidden_units)
        self.drop1 = nn.Dropout(p=dropout_p)
        self.fc2   = nn.Linear(hidden_units, hidden_units)
        self.drop2 = nn.Dropout(p=dropout_p)
        self.out   = nn.Linear(hidden_units, n_classes)

    def forward(self, x):
        x = F.relu(self.conv(x))       # Conv → ReLU
        x = self.pool(x)               # MaxPool 7×7×7
        x = x.view(x.size(0), -1)     # Flatten
        x = self.drop1(F.relu(self.fc1(x)))
        x = self.drop2(F.relu(self.fc2(x)))
        return F.log_softmax(self.out(x), dim=1)

    def init_from_sae(self, weight, bias):
        """Transfer SAE encoder weights to the conv layer."""
        with torch.no_grad():
            self.conv.weight.copy_(weight)
            self.conv.bias.copy_(bias)


# ── Architecture check ────────────────────────────────────────────────────────
cnn_chk = CNN3D(n_classes=3)
dummy   = torch.zeros(2, 1, *TARGET_SHAPE)
out_chk = cnn_chk(dummy)
print(f"CNN input     : {dummy.shape}")
print(f"CNN output    : {out_chk.shape}")
print(f"flat_size     : {cnn_chk.flat_size:,}")
total_p = sum(p.numel() for p in cnn_chk.parameters())
print(f"Total params  : {total_p:,}")
del cnn_chk, dummy, out_chk
print("✅ CNN OK")

## Cell 9 — Metrics (Paper Section 4.3, Equations 2–4)

In [ ]:
def compute_binary_metrics(y_true, y_pred):
    """
    Accuracy (Eq.2), Sensitivity (Eq.3), Specificity (Eq.4).
    Positive class = label 0  (e.g. AD in AD:NC task).
    """
    yt = np.array(y_true)
    yp = np.array(y_pred)
    TP = np.sum((yt == 0) & (yp == 0))
    TN = np.sum((yt == 1) & (yp == 1))
    FP = np.sum((yt == 1) & (yp == 0))
    FN = np.sum((yt == 0) & (yp == 1))
    acc = (TP + TN) / (TP + TN + FP + FN + 1e-8) * 100
    sen = TP / (TP + FN + 1e-8) * 100
    spe = TN / (TN + FP + 1e-8) * 100
    return float(acc), float(sen), float(spe)


def compute_multiclass_acc(y_true, y_pred):
    """Overall accuracy for the 3-class task."""
    return float(np.mean(np.array(y_true) == np.array(y_pred)) * 100)


def run_epoch(model, loader, loss_fn, optimizer=None, train=True):
    """
    One forward pass over `loader`.
    If train=True, also does backprop (optimizer must be provided).
    Returns (mean_loss, y_true_list, y_pred_list).
    """
    model.train() if train else model.eval()
    total_loss, yt, yp = 0.0, [], []
    ctx = torch.enable_grad() if train else torch.no_grad()

    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            out  = model(x)
            loss = loss_fn(out, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            yp.extend(out.argmax(1).cpu().numpy())
            yt.extend(y.cpu().numpy())

    return total_loss / max(len(loader), 1), yt, yp


print("Metric & training functions defined. ✅")

## Cell 10 — Cross-validation runner

In [ ]:
def run_task(task_subjects, task_labels, n_classes, task_name, is_binary=True):
    """
    Full cross-validation for one classification task.

    Parameters
    ----------
    task_subjects : list of subject dicts (label already remapped for this task)
    task_labels   : list of int labels matching task_subjects
    n_classes     : 2 (binary) or 3 (multi-class)
    task_name     : string e.g. 'AD:NC'
    is_binary     : if True, compute SEN and SPE

    Returns dict with mean±std for ACC, SEN, SPE.
    """
    print(f"\n{'='*65}")
    print(f"  Task : {task_name}  |  Subjects : {len(task_subjects)}  "
          f"|  Classes : {n_classes}")
    # Show modality split
    fbb_n  = sum(1 for s in task_subjects if s.get('modality') == 'FBB')
    av45_n = len(task_subjects) - fbb_n
    print(f"  Modality split: {fbb_n} FBB + {av45_n} AV45")
    print(f"{'='*65}")

    dataset    = FusedDataset(task_subjects)
    labels_arr = np.array(task_labels)

    # Reduce folds for very small datasets
    n_folds = N_FOLDS
    min_class = min(Counter(task_labels).values())
    if min_class < n_folds:
        n_folds = max(min_class, 3)
        print(f"  [NOTE] Reduced to {n_folds}-fold "
              f"(smallest class has only {min_class} subjects)")

    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=42)

    all_acc, all_sen, all_spe = [], [], []

    for fold, (tr_idx, te_idx) in enumerate(
            skf.split(np.zeros(len(labels_arr)), labels_arr)):

        print(f"\n── Fold {fold+1}/{n_folds}  "
              f"(train={len(tr_idx)}, test={len(te_idx)}) ──")

        tr_loader = DataLoader(
            Subset(dataset, tr_idx),
            batch_size=BATCH_SIZE, shuffle=True,
            num_workers=2, pin_memory=(DEVICE=='cuda'))
        te_loader = DataLoader(
            Subset(dataset, te_idx),
            batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=(DEVICE=='cuda'))

        # ── Step 1: Pre-train Sparse Autoencoder ──────────────────────────────
        print("  [1/3] Pre-training Sparse Autoencoder...")
        sae    = train_sae(tr_loader, verbose=True)
        w, b   = sae.encoder_weights()

        # ── Step 2: Build 3D CNN with SAE-initialised weights ─────────────────
        print("  [2/3] Building 3D CNN (SAE-initialised)...")
        model  = CNN3D(n_classes=n_classes).to(DEVICE)
        model.init_from_sae(w.to(DEVICE), b.to(DEVICE))
        opt    = torch.optim.Adam(model.parameters(), lr=LR)
        loss_fn = nn.NLLLoss()

        # ── Step 3: Train CNN for EPOCHS epochs ───────────────────────────────
        print(f"  [3/3] Training CNN ({EPOCHS} epochs)...")
        for epoch in range(EPOCHS):
            tr_loss, _, _     = run_epoch(model, tr_loader, loss_fn,
                                          opt, train=True)
            te_loss, yt, yp   = run_epoch(model, te_loader, loss_fn,
                                          train=False)
            # Print every 10 epochs (matches paper reporting interval)
            if (epoch + 1) % 10 == 0:
                if is_binary:
                    acc, _, _ = compute_binary_metrics(yt, yp)
                else:
                    acc = compute_multiclass_acc(yt, yp)
                print(f"    Epoch {epoch+1:3d}/{EPOCHS}  "
                      f"tr={tr_loss:.4f}  te={te_loss:.4f}  "
                      f"ACC={acc:.2f}%")

        # ── Final evaluation ──────────────────────────────────────────────────
        _, yt_f, yp_f = run_epoch(model, te_loader, loss_fn, train=False)

        if is_binary:
            acc, sen, spe = compute_binary_metrics(yt_f, yp_f)
            all_acc.append(acc); all_sen.append(sen); all_spe.append(spe)
            print(f"  ✔ Fold {fold+1}: ACC={acc:.2f}%  "
                  f"SEN={sen:.2f}%  SPE={spe:.2f}%")
        else:
            acc = compute_multiclass_acc(yt_f, yp_f)
            all_acc.append(acc)
            print(f"  ✔ Fold {fold+1}: ACC={acc:.2f}%")

    # ── Summary ───────────────────────────────────────────────────────────────
    print(f"\n{'='*65}")
    print(f"  FINAL — {task_name}")
    print(f"  ACC : {np.mean(all_acc):.2f} ± {np.std(all_acc):.2f}%")
    if is_binary:
        print(f"  SEN : {np.mean(all_sen):.2f} ± {np.std(all_sen):.2f}%")
        print(f"  SPE : {np.mean(all_spe):.2f} ± {np.std(all_spe):.2f}%")
    print(f"{'='*65}")

    return {
        'task': task_name,
        'acc' : (np.mean(all_acc), np.std(all_acc)),
        'sen' : (np.mean(all_sen), np.std(all_sen)) if is_binary else (None, None),
        'spe' : (np.mean(all_spe), np.std(all_spe)) if is_binary else (None, None),
    }


print("Cross-validation runner ready. ✅")

## Cell 11 — Task 1: AD vs CN (Paper Table 2)
Uses: `ADNI_FUSED_FBB_AD` + `ADNI_FUSED_OUTPUT_AD` (all AD subjects)  
vs `ADNI_FUSED_OUTPUT_CN` (all CN subjects)

In [ ]:
# AD subjects come from both FBB and OUTPUT folders (label=0)
# CN subjects come from OUTPUT only (label=2)
ad_nc_raw = [s for s in all_subjects if s['label'] in [0, 2]]

# Remap: AD→0 (positive class), CN→1 (negative class)
ad_nc_subs = []
for s in ad_nc_raw:
    s2 = dict(s)
    s2['label'] = 0 if s['label'] == 0 else 1
    ad_nc_subs.append(s2)

ad_nc_labels = [s['label'] for s in ad_nc_subs]

print(f"AD:NC — AD={ad_nc_labels.count(0)}  CN={ad_nc_labels.count(1)}")
ad_fbb  = sum(1 for s in ad_nc_subs if s['label']==0 and s['modality']=='FBB')
ad_av45 = sum(1 for s in ad_nc_subs if s['label']==0 and s['modality']=='AV45')
print(f"  AD breakdown: {ad_fbb} FBB + {ad_av45} AV45")
print(f"Paper target : ACC~93.21%  SEN~91.43%  SPE~95.42%\n")

result_ad_nc = run_task(
    ad_nc_subs, ad_nc_labels,
    n_classes=2, task_name="AD:NC", is_binary=True
)

## Cell 12 — Task 2: MCI vs CN (Paper Table 3)
Uses: `ADNI_FUSED_FBB_MCI` + `ADNI_FUSED_OUTPUT_MCI` (all MCI subjects)  
vs `ADNI_FUSED_OUTPUT_CN` (all CN subjects)

In [ ]:
mci_nc_raw = [s for s in all_subjects if s['label'] in [1, 2]]

# Remap: MCI→0 (positive), CN→1 (negative)
mci_nc_subs = []
for s in mci_nc_raw:
    s2 = dict(s)
    s2['label'] = 0 if s['label'] == 1 else 1
    mci_nc_subs.append(s2)

mci_nc_labels = [s['label'] for s in mci_nc_subs]

print(f"MCI:NC — MCI={mci_nc_labels.count(0)}  CN={mci_nc_labels.count(1)}")
mci_fbb  = sum(1 for s in mci_nc_subs if s['label']==0 and s['modality']=='FBB')
mci_av45 = sum(1 for s in mci_nc_subs if s['label']==0 and s['modality']=='AV45')
print(f"  MCI breakdown: {mci_fbb} FBB + {mci_av45} AV45")
print(f"Paper target : ACC~86.52%  SEN~94.34%  SPE~81.64%\n")

result_mci_nc = run_task(
    mci_nc_subs, mci_nc_labels,
    n_classes=2, task_name="MCI:NC", is_binary=True
)

## Cell 13 — Task 3: AD vs MCI (Paper Table 4)
Uses all AD subjects (FBB + OUTPUT) vs all MCI subjects (FBB + OUTPUT)

In [ ]:
ad_mci_raw = [s for s in all_subjects if s['label'] in [0, 1]]

# Labels: AD=0 (positive), MCI=1 (negative) — no remap needed
ad_mci_subs   = [dict(s) for s in ad_mci_raw]
ad_mci_labels = [s['label'] for s in ad_mci_subs]

print(f"AD:MCI — AD={ad_mci_labels.count(0)}  MCI={ad_mci_labels.count(1)}")
print(f"Paper target : ACC~85.63%  SEN~81.21%  SPE~95.54%\n")

result_ad_mci = run_task(
    ad_mci_subs, ad_mci_labels,
    n_classes=2, task_name="AD:MCI", is_binary=True
)

## Cell 14 — Task 4: AD vs MCI vs CN (Paper Table 5)
Uses all subjects from all 5 folders

In [ ]:
all_subs_copy = [dict(s) for s in all_subjects]
all_lbl_copy  = [s['label'] for s in all_subs_copy]

print(f"AD:MCI:CN — AD={all_lbl_copy.count(0)}  "
      f"MCI={all_lbl_copy.count(1)}  CN={all_lbl_copy.count(2)}")
print(f"Total: {len(all_subs_copy)} subjects")
print(f"Paper target : ACC~87.67%\n")

result_3class = run_task(
    all_subs_copy, all_lbl_copy,
    n_classes=3, task_name="AD:MCI:NC", is_binary=False
)

## Cell 15 — Final results summary

In [ ]:
paper = {
    'AD:NC'    : (93.21, 91.43, 95.42),
    'MCI:NC'   : (86.52, 94.34, 81.64),
    'AD:MCI'   : (85.63, 81.21, 95.54),
    'AD:MCI:NC': (87.67, None,  None),
}

print("\n" + "═"*70)
print("  COMPLETE RESULTS vs PAPER (Kong et al., 2022)")
print("═"*70)
print(f"{'Task':<12} {'Ours ACC':>14} {'Paper ACC':>10} "
      f"{'Ours SEN':>12} {'Paper SEN':>10}")
print("─"*70)

for res in [result_ad_nc, result_mci_nc, result_ad_mci, result_3class]:
    task          = res['task']
    acc_m, acc_s  = res['acc']
    sen_m, _      = res['sen']
    pt            = paper[task]

    ours_acc = f"{acc_m:.2f}±{acc_s:.2f}%"
    ours_sen = f"{sen_m:.2f}%" if sen_m is not None else "     —"
    p_acc    = f"{pt[0]:.2f}%"
    p_sen    = f"{pt[1]:.2f}%" if pt[1] else "     —"

    print(f"{task:<12} {ours_acc:>14} {p_acc:>10} {ours_sen:>12} {p_sen:>10}")

print("═"*70)
print("\nDataset used:")
fbb_total  = sum(1 for s in all_subjects if s['modality']=='FBB')
av45_total = sum(1 for s in all_subjects if s['modality']=='AV45')
for label, name in enumerate(CLASS_NAMES):
    fbb_c  = sum(1 for s in all_subjects if s['label']==label and s['modality']=='FBB')
    av45_c = sum(1 for s in all_subjects if s['label']==label and s['modality']=='AV45')
    print(f"  {name}: {fbb_c+av45_c} ({fbb_c} FBB + {av45_c} AV45)")
print(f"  Total: {len(all_subjects)} ({fbb_total} FBB + {av45_total} AV45)")
print("\nNote: Results may differ from paper due to smaller dataset (97 vs 370).")